In [ ]:
import os

# Always run this first — sets working directory to the project root
# The notebook lives in notebooks/, so one level up is the root
os.chdir("..")

# Confirm it worked
print("Working directory:", os.getcwd())


In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


In [ ]:
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases"
    "/heart-disease/processed.cleveland.data"
)
column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "target"
]
df = pd.read_csv(url, names=column_names, na_values='?').dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")


In [ ]:
mlflow.set_experiment("heart-disease-classification")

In [ ]:
def run_logistic_regression(C, max_iter, run_name):
    """Train a logistic regression and log everything to MLflow."""
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("solver", "lbfgs")

        # Train
        model = LogisticRegression(
            C=C, max_iter=max_iter, solver='lbfgs', random_state=42
        )
        model.fit(X_train_scaled, y_train)

        # Evaluate
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        print(f"{run_name}: accuracy={accuracy:.4f}, roc_auc={roc_auc:.4f}")

# Run two new logistic regression variants
run_logistic_regression(C=0.01, max_iter=200, run_name='lr-C0.01')
run_logistic_regression(C=10.0, max_iter=200, run_name='lr-C10.0')


In [ ]:
def run_gradient_boosting(n_estimators, learning_rate, max_depth, run_name):
    """Train a gradient boosting model and log everything to MLflow."""
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_param("model_type", "GradientBoosting")
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_depth", max_depth)

        # Train — note: GradientBoosting uses unscaled data fine,
        # but we use scaled for consistency across experiments
        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)

        # Evaluate
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        print(f"{run_name}: accuracy={accuracy:.4f}, roc_auc={roc_auc:.4f}")

# Run two gradient boosting variants
run_gradient_boosting(n_estimators=100, learning_rate=0.1, max_depth=3, run_name='gb-100-lr0.1')
run_gradient_boosting(n_estimators=200, learning_rate=0.05, max_depth=4, run_name='gb-200-lr0.05')

